# Notebook 07: Baseline Traffic Signal Controller

**Purpose:** Run a fixed-time traffic light baseline simulation run and record mobility and emission performance metrics.

The baseline policy uses a conventional fixed-time round-robin policy:
- Green NS: 30s
- Yellow NS: 4s
- Green EW: 30s
- Yellow EW: 4s


In [ ]:
import traci
import numpy as np
import json
from pathlib import Path
import sys

sys.path.append(str(Path("..").resolve()))
from backend.simulation.traci_client import traci_client
from backend.simulation.sumo_manager import sumo_manager
from backend.simulation.traffic_lights import TrafficLightsManager
from backend.simulation.traffic_state import TrafficState
from backend.simulation.emission_collector import EmissionCollector


### Run Simulation with Fixed-Time Signal Policy

In [ ]:
binary = sumo_manager.get_binary_path(force_gui=False)
sumocfg = sumo_manager.get_default_config_path("evaluation")

print(f"Connecting to SUMO: {binary}")
traci_client.connect(binary, sumocfg, 1.0, label="fixed_time_baseline")

tls_ids = TrafficLightsManager.get_tls_ids()
tls_id = tls_ids[0] if tls_ids else "J1"

# Metrics list
co2_steps = []
wait_steps = []
speed_steps = []
nox_steps = []
fuel_steps = []

# Fixed-time parameters
phase_durations = [30, 4, 30, 4]
current_phase = 0
time_in_phase = 0

print(f"Starting simulation for 1000 steps on TLS: {tls_id}")
for step in range(1000):
    traci_client.step()
    
    # Manage fixed-time traffic lights state machine
    time_in_phase += 1
    if time_in_phase >= phase_durations[current_phase]:
        current_phase = (current_phase + 1) % 4
        time_in_phase = 0
        TrafficLightsManager.set_phase(tls_id, current_phase)
        
    # Collect step metrics
    emissions = EmissionCollector.get_system_emissions()
    co2_steps.append(emissions["co2"])
    nox_steps.append(emissions["nox"])
    fuel_steps.append(emissions["fuel"])
    
    vehicles = TrafficState.get_active_vehicles()
    if vehicles:
        avg_wait = np.mean([v["waiting_time"] for v in vehicles])
        avg_speed = np.mean([v["speed"] for v in vehicles])
    else:
        avg_wait = 0.0
        avg_speed = 0.0
        
    wait_steps.append(avg_wait)
    speed_steps.append(avg_speed)

traci_client.close()


### Compile & Save Baseline Performance Metrics

In [ ]:
fixed_time_metrics = {
    "total_co2_mg": float(np.sum(co2_steps)),
    "average_co2_mg_s": float(np.mean(co2_steps)),
    "total_nox_mg": float(np.sum(nox_steps)),
    "total_fuel_ml": float(np.sum(fuel_steps)),
    "average_waiting_time_s": float(np.mean(wait_steps)),
    "average_speed_kmh": float(np.mean(speed_steps))
}

print("Fixed-Time Baseline Performance:")
print(json.dumps(fixed_time_metrics, indent=4))

# Save metrics JSON
with open("../reports/fixed_time_metrics.json", "w") as f:
    json.dump(fixed_time_metrics, f, indent=4)
